In [1]:
# Import libraries

import json # For working with JSON data
import os # For interacting with the operating system
from dotenv import load_dotenv # For loading environment variables from a .env file
import shutil # For High-level file operations
from pathlib import Path # For object-oriented filesystem paths
from typing import Literal # For type hinting with Literal types

import httpx # For making HTTP requests
import pandas as pd # For data manipulation and analysis
import plotly.express as px # For creating interactive plots

from langchain.agents import create_agent # For creating agents in LangChain
from langchain.agents import create_agent # For creating agents in LangChain
from langchain.agents.middleware import ToolRetryMiddleware # For retrying tools in LangChain agents
from langchain_core.tools import tool # For defining tools in LangChain
from langchain_mcp_adapters.client import MultiServerMCPClient # For interacting with multiple MCP servers
from langchain_openai import ChatOpenAI # For using OpenAI chat models in LangChain

from langfuse import Evaluation, get_client # For working with Langfuse evaluations and client
from langfuse.langchain import CallbackHandler # For handling Langfuse callbacks in LangChain

from week2day1_payout import Benefit, Line, Payout, work_out_payout # For handling payout calculations and related classes


### Connect to OpenRouter and LangFuse

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
langfuse = get_client()

if openrouter_api_key:
    print("OpenRouter API Key found and good to go")
else:
    print("OpenRouter API Key not found")

if langfuse.auth_check():
    print("Langfuse loaded and good to go")
else:
    print("Langfuse not loaded")

OpenRouter API Key found and good to go
Langfuse loaded and good to go


## Part 1: The even harder claims

In [3]:
EVEN_HARDER =Path("../../../data/evals/even_harder")

def load(name):
    """Read one of the JSON files of the even harder claims.. """
    return json.loads((EVEN_HARDER / name).read_text(encoding='utf-8'))

claims = load('claims.json')
labels ={decision['claim_id']: decision for decision in load('decisions.json')}
policies = {policy['policy_id']: policy for policy in load('policies.json')}
policy_wording = Path('../../..//data/policy_wording.md').read_text(encoding='utf-8')

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

25 claims, 25 labels, 25 policies


In [4]:
rows =[]
for claim in claims:
    label = labels[claim['claim_id']]
    rows.append({
        'claim_id': claim['claim_id'],
        'summary': claim['summary'],
        'amount_claimed': claim['amount_claimed'],
        'decision': label['decision'],
        'payout': label['payout']
    })

    data = pd.DataFrame(rows)
    data

### A lost bag in Tokyo

In [8]:
tokyo = [claim for claim in claims if claim['claim_id'] == 'CLM-1307'][0]

print(tokyo['story'])
pd.DataFrame(tokyo['items'])

My checked bag never arrived in Tokyo. Skylark searched for a month, declared it lost and paid me $400. Everything in it is listed, with receipts for anything over $100.


,description,category,amount
0,Suitcase,luggage,189.00
1,Halden Arc,electronics,899.00
2,Lightweight rain jacket,clothing,165.00
3,Two merino T-shirts,clothing,118.00
4,"Hiking shoes, one pair",clothing,139.95
5,Packing cubes,luggage,54.20
6,Charging cables and adapters,electronics,46.80


In [9]:
decision = labels['CLM-1307']

print(decision['reason'])
pd.DataFrame([decision['calculation']])

The carrier declared the bag lost. The Halden Arc is a pocket projector, which is not on the Electronics Plus list, so it counts at the $300 per-item limit on Premium, and the other lines are within the limit. The items come to $1,012.95, less the carrier's $400 and the $50 deductible, leaving $562.95, inside the $1,500 maximum.


,0,1,2,3,4,5,6,7,8
0,"{'step': 'Suitcase', 'amount': 189.0}","{'step': 'Halden Arc pocket projector, limited...","{'step': 'Lightweight rain jacket', 'amount': ...","{'step': 'Two merino T-shirts', 'amount': 118.0}","{'step': 'Hiking shoes', 'amount': 139.95}","{'step': 'Packing cubes', 'amount': 54.2}","{'step': 'Charging cables and adapters', 'amou...",{'step': 'Less what others have paid or refund...,"{'step': 'Less the deductible', 'amount': -50.0}"


### The Knowledge Base

In [10]:
knowledge = json.loads(Path('../../../data/knowledge.json').read_text(encoding='utf-8'))

pd.set_option('display.max_colwidth', None)
pd.DataFrame(knowledge)

,topic,fact
0,events,"National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east of the Leeward Islands. It is forecast to become a hurricane on July 11 and to pass over the Bahamas on July 12 and 13."
1,events,"National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in the western Caribbean. It is expected to become a hurricane and to reach the coast of the Yucatan Peninsula near Cancun on August 21."
2,events,"Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 hour strike starting at midnight on July 14. Most flights to and from Lisbon and Porto are expected to be canceled on July 14 and 15."
3,events,"Meridian Air press release, August 5, 2026: The union representing our cabin crew has called a strike for August 12 and 13. We expect to cancel about half of our flights on those two days and will rebook affected passengers."
4,events,"North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named. Gales are expected across Ireland, Wales and western Scotland from the morning of September 4, and ferry crossings on the Irish Sea are likely to be canceled."
5,events,"Rome, July 3, 2026: Italy's rail unions have called a 24 hour national rail strike from 9 pm on July 21 to 9 pm on July 22. Most high-speed and regional trains will not run."
6,events,"National Hurricane Center, June 30, 2026: Tropical Storm Nadia has formed in the southern Gulf of Mexico and is expected to come ashore in Tamaulipas, Mexico, on July 2."
7,events,"Northwind Airlines press release, July 15, 2026: The pilots' strike planned for July 24 has been called off after an agreement with the union. All flights will operate as scheduled."
8,events,"Japan Meteorological Agency, August 24, 2026: Typhoon Kaori is expected to pass close to Okinawa on August 27 and 28. Flights and ferries in the Ryukyu Islands are likely to be suspended."
9,events,"Athens, August 19, 2026: Greek ferry workers have called a 24 hour strike for August 27. Ferries between Piraeus and the islands will stay in port."


## Part 2: A memory over MCP

In [11]:
MEMORY = Path('qdrant_data').resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

In [12]:
MEMORY_ENV ={'QDRANT_LOCAL_PATH': str(MEMORY), 'COLLECTION_NAME': 'memory'}
parameters={
    'transport': 'stdio',
    'command': 'uvx',
    'args': ['mcp-server-qdrant@0.8.1'],
    'env': MEMORY_ENV
}
client = MultiServerMCPClient({
    'memory': parameters
})
memory_tools = await client.get_tools()
for memory_tool in memory_tools:
    print(f"{memory_tool.name}: {memory_tool.description}\n")

qdrant-find: Look up memories in Qdrant. Use this tool when you need to: 
 - Find memories by their content 
 - Access memories for further analysis 
 - Get some personal information about the user

qdrant-store: Keep the memory for later use, when you are asked to remember something.



### An agent with memory

In [14]:
MEMORY_INSTRUCTIONS = """ You have a long-term memory. When you are told a fact, store it
in your memory word for word. Whe you are asked a question, search your memory and answer
from what you find. If your memory has nothing on it, say that you don't know.
"""

http_client = httpx.AsyncClient(
    limits=httpx.Limits(max_keepalive_connections=0)
)
luna = ChatOpenAI(
    model='openai/gpt-6-luna',
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    reasoning_effort='low',
    timeout=120,
    max_retries=5,
    http_async_client=http_client
)
assistant = create_agent(
    model=luna,
    tools=memory_tools,
    system_prompt=MEMORY_INSTRUCTIONS,
    middleware=[ToolRetryMiddleware(max_retries=5)]
)

In [16]:
async def ask(message):
    """ Send a message to the assistant in a new conversation and return its reply. """
    result = await assistant.ainvoke({
        'messages': [
            {
                'role': 'user',
                'content': message
            }
        ]
    })
    return result['messages'][-1].text
print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

I don’t know. I couldn’t find anything about the Halden Arc in my memory.
I don’t know.


## Part 3: Teaching the agent

In [17]:
for entry in knowledge:
    await ask(f"Please remember this: {entry['fact']}")
    print(f"Told: {entry['fact'][:80]} ...")

Told: National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east  ...
Told: National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in  ...
Told: Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 ho ...
Told: Meridian Air press release, August 5, 2026: The union representing our cabin cre ...
Told: North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named. ...
Told: Rome, July 3, 2026: Italy's rail unions have called a 24 hour national rail stri ...
Told: National Hurricane Center, June 30, 2026: Tropical Storm Nadia has formed in the ...
Told: Northwind Airlines press release, July 15, 2026: The pilots' strike planned for  ...
Told: Japan Meteorological Agency, August 24, 2026: Typhoon Kaori is expected to pass  ...
Told: Athens, August 19, 2026: Greek ferry workers have called a 24 hour strike for Au ...
Told: Paris, May 29, 2026: French air traffic controllers have called a strike for Jun ...

In [18]:
print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

The Halden Arc is a pocket projector that can project an image up to 100 inches from a phone or laptop.
Tropical Storm Marisol was named on July 9, 2026.


In [19]:
search = [memory_tool for memory_tool in memory_tools if memory_tool.name =='qdrant-find'][0]
results = await search.ainvoke({
    'query': 'a gadget for watching films on a wall'
})
for line in json.loads(results[0]['text']):
    print(line)

Results for the query 'a gadget for watching films on a wall'
<entry><content>Halden Arc: a pocket projector that throws a picture of up to 100 inches from a phone or laptop. List price $899.</content><metadata></metadata></entry>
<entry><content>Vantor Halo: a titanium smart ring that tracks sleep, heart rate and activity. List price $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 3: an e-reader with a 7 inch color E Ink screen and a battery that lasts six weeks. List price $329.</content><metadata></metadata></entry>
<entry><content>Corvane Lumen X1: the earlier Corvane smartwatch, released in 2024 at a list price of $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 2: the earlier Lumora e-reader, with a 6 inch black and white screen. List price $199.</content><metadata></metadata></entry>
<entry><content>Halden Beam: a portable Bluetooth speaker with a built-in lamp. List price $199.</content><metadata></metadata></entry>
<entry><con

## Part 4: Experiments in Langfuse

In [ ]:
INSTRUCTIONS = """ You are a claims handler at UnderCover Travel Insurance. Decide the claim
you are given, using the policy wording below and the customers policy record."""

